In [1]:
from pathlib import Path
import random
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from tqdm import tqdm

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
from src.config import DATA_PROCESSED, FEMTO_DIR, FEMTO_FS, OVERLAP, REPORTS_DIR, SEED, WINDOW_SIZE
from src.data.loader import load_femto_bearing
from src.features.extraction import extract_bearing_features

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
FIGURES_DIR = REPORTS_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

In [3]:
femto_root = FEMTO_DIR / "10. FEMTO Bearing"
canonical_roots = [
    femto_root / "Training_set" / "Learning_set",
    femto_root / "Validation_Set" / "Full_Test_Set",
]
canonical_bearing_paths = sorted(
    bearing_path for root in canonical_roots for bearing_path in root.iterdir() if bearing_path.is_dir()
)

In [4]:
feature_frames = []

for bearing_path in tqdm(canonical_bearing_paths, desc="FEMTO feature extraction"):
    bearing_frame = load_femto_bearing(bearing_path)
    bearing_features = extract_bearing_features(bearing_frame, FEMTO_FS, WINDOW_SIZE, OVERLAP)
    bearing_features["bearing_id"] = bearing_path.name
    feature_frames.append(bearing_features)

femto_features = pd.concat(feature_frames, ignore_index=True)

FEMTO feature extraction:   0%|          | 0/17 [00:00<?, ?it/s]

FEMTO feature extraction:   6%|▌         | 1/17 [00:11<02:59, 11.19s/it]

FEMTO feature extraction:  12%|█▏        | 2/17 [00:23<02:59, 11.95s/it]

FEMTO feature extraction:  18%|█▊        | 3/17 [00:36<02:53, 12.37s/it]

FEMTO feature extraction:  24%|██▎       | 4/17 [00:47<02:33, 11.84s/it]

...

FEMTO feature extraction:  82%|████████▏ | 14/17 [02:49<00:31, 10.46s/it]

FEMTO feature extraction:  88%|████████▊ | 15/17 [02:52<00:16,  8.42s/it]

FEMTO feature extraction:  94%|█████████▍| 16/17 [02:54<00:06,  6.25s/it]

FEMTO feature extraction: 100%|██████████| 17/17 [02:56<00:00,  5.05s/it]

FEMTO feature extraction: 100%|██████████| 17/17 [02:56<00:00, 10.37s/it]

In [5]:
feature_path = DATA_PROCESSED / "femto_features_all_bearings.parquet"
femto_features.to_parquet(feature_path, index=False)

print(f"shape: {femto_features.shape}")
print(f"columns: {femto_features.columns.tolist()}")
print(f"missing values: {int(femto_features.isna().sum().sum())}")
print(femto_features.groupby("bearing_id").size().sort_index())

shape: (99556, 47)
columns: ['ch_h_rms', 'ch_h_mean', 'ch_h_std', 'ch_h_kurtosis', 'ch_h_skewness', 'ch_h_peak', 'ch_h_crest_factor', 'ch_h_peak_to_peak', 'ch_h_shape_factor', 'ch_h_spectral_mean', 'ch_h_spectral_std', 'ch_h_spectral_kurtosis', 'ch_h_dominant_freq', 'ch_h_total_power', 'ch_h_band_1_energy', 'ch_h_band_2_energy', 'ch_h_band_3_energy', 'ch_h_band_4_energy', 'ch_h_env_mean', 'ch_h_env_std', 'ch_h_env_kurtosis', 'ch_h_env_rms', 'ch_v_rms', 'ch_v_mean', 'ch_v_std', 'ch_v_kurtosis', 'ch_v_skewness', 'ch_v_peak', 'ch_v_crest_factor', 'ch_v_peak_to_peak', 'ch_v_shape_factor', 'ch_v_spectral_mean', 'ch_v_spectral_std', 'ch_v_spectral_kurtosis', 'ch_v_dominant_freq', 'ch_v_total_power', 'ch_v_band_1_energy', 'ch_v_band_2_energy', 'ch_v_band_3_energy', 'ch_v_band_4_energy', 'ch_v_env_mean', 'ch_v_env_std', 'ch_v_env_kurtosis', 'ch_v_env_rms', 'file_idx', 'window_idx', 'bearing_id']
missing values: 0
bearing_id
Bearing1_1    11212
Bearing1_2     3484
Bearing1_3     9500
Bearing1_4

In [6]:
numeric_columns = femto_features.select_dtypes(include=np.number).columns
feature_columns = numeric_columns.drop(["file_idx", "window_idx"])
top_variance_features = femto_features[feature_columns].var().sort_values(ascending=False).head(20).index

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(femto_features[top_variance_features].corr(), cmap="vlag", center=0, ax=ax)
ax.set_title("Top-Variance FEMTO Feature Correlation")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "femto_feature_correlation_top20_variance.png", dpi=150)
plt.close(fig)

In [7]:
bearing1_1_features = femto_features[femto_features["bearing_id"] == "Bearing1_1"]
trend_columns = ["ch_h_rms", "ch_h_kurtosis", "ch_h_env_kurtosis"]
feature_trends = bearing1_1_features.groupby("file_idx")[trend_columns].mean()

fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=True)
for axis, column in zip(axes, trend_columns):
    axis.plot(feature_trends.index, feature_trends[column], linewidth=1.2, label=column)
    axis.set_ylabel(column)
    axis.legend()
axes[-1].set_xlabel("File index (× 10s)")
fig.suptitle("Bearing1_1 Horizontal Feature Trends")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "femto_bearing1_1_feature_trends.png", dpi=150)
plt.close(fig)

The most visible degradation indicators for Bearing1_1 are expected to be energy and impulsiveness features: RMS should rise as vibration severity increases, while kurtosis and envelope kurtosis should react to impacts and non-stationary bearing damage. Highly correlated feature blocks are useful for interpretation but also mean tree or margin baselines may need scaling and careful threshold freezing.